# Stage 7 — Retriever

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Reload the saved FAISS vector store, wrap it as a LangChain retriever, and test it with several realistic questions — including one with no good match, to see how it behaves.

**Before running:** upload `faiss_index.zip` (from Stage 6) to this Colab session.

## Cell 1 — Install packages

In [1]:
!pip install -q faiss-cpu langchain-community langchain-huggingface sentence-transformers langchain-core

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.8/18.8 MB 75.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 78.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 50.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 5.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


## Cell 2 — Unzip and reload the saved vector store

We unzip `faiss_index.zip`, load the same embedding model used to build it (required — FAISS needs to know how to embed new queries in the same vector space), and load the index with `FAISS.load_local`. `allow_dangerous_deserialization=True` is required because loading a FAISS index involves unpickling files — safe here since we created this file ourselves in Stage 6.

In [2]:
import zipfile
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS

with zipfile.ZipFile("faiss_index.zip", "r") as zip_ref:
    zip_ref.extractall("faiss_index")

embedding_model = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

vectorstore = FAISS.load_local(
    "faiss_index",
    embedding_model,
    allow_dangerous_deserialization=True
)

print("Vector store reloaded. Total vectors:", vectorstore.index.ntotal)

/tmp/ipykernel_1547/316989208.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.vectorstores import FAISS


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Vector store reloaded. Total vectors: 20971


## Cell 3 — Create the retriever

`as_retriever` wraps the vector store in a standard LangChain interface: instead of calling FAISS-specific methods, we can now just call `.invoke(question)` and get documents back. `search_kwargs={"k": 3}` sets top-k = 3: return the 3 most relevant documents per question.

In [3]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("Retriever created with top-k =", retriever.search_kwargs["k"])

Retriever created with top-k = 3


## Cell 4 — Test the retriever on one question

We call `retriever.invoke(question)` and print the retrieved papers' titles, topics, and a short snippet — this is exactly what will feed into the LLM's context in Stage 8.

In [4]:
def show_results(question, docs):
    print(f"QUESTION: {question}\n")
    for i, doc in enumerate(docs, start=1):
        print(f"--- Result {i} ---")
        print("Title:", doc.metadata["title"])
        print("Topics:", doc.metadata["topics"])
        print("Snippet:", doc.page_content[:200], "...")
        print()

question = "What research has been done on transformer efficiency?"
docs = retriever.invoke(question)
show_results(question, docs)

QUESTION: What research has been done on transformer efficiency?

--- Result 1 ---
Title: Leveraging Sensory Data in Estimating Transformer Lifetime
Topics: Computer Science
Snippet: Leveraging Sensory Data in Estimating Transformer Lifetime

  Transformer lifetime assessments plays a vital role in reliable operation of
power systems. In this paper, leveraging sensory data, an app ...

--- Result 2 ---
Title: The Integral Transform of N.I.Akhiezer
Topics: Mathematics
Snippet: The Integral Transform of N.I.Akhiezer

  We study the integral transform which appeared in a different form in
Akhiezer's textbook "Lectures on Integral Transforms". ...

--- Result 3 ---
Title: Carnot Efficiency of Publication
Topics: Computer Science
Snippet: Carnot Efficiency of Publication

  This paper analyzes publication efficiency in terms of Hirsch-index or
h-index and total citations, with an analogy to the Carnot efficiency used in
thermodynamics. ...



## Cell 5 — See the actual relevance scores

The retriever itself doesn't expose scores, so we call the vector store directly with `similarity_search_with_score` for the same question.

**Important:** FAISS's default score here is an L2 (Euclidean) **distance**, not a percentage — so **lower means more similar**, the opposite of the cosine similarity numbers from Stage 5. We print it explicitly so this doesn't cause confusion later.

In [5]:
results_with_scores = vectorstore.similarity_search_with_score(question, k=3)

print(f"QUESTION: {question}\n")
for doc, score in results_with_scores:
    print(f"Distance: {score:.4f}  (lower = more similar)  |  Title: {doc.metadata['title']}")

QUESTION: What research has been done on transformer efficiency?

Distance: 0.9856  (lower = more similar)  |  Title: Leveraging Sensory Data in Estimating Transformer Lifetime
Distance: 1.2952  (lower = more similar)  |  Title: The Integral Transform of N.I.Akhiezer
Distance: 1.3255  (lower = more similar)  |  Title: Carnot Efficiency of Publication


## Cell 6 — Test with several more realistic questions

We try a range of questions across different topics in our dataset (Computer Science, Physics, Statistics, Quantitative Finance), to see the retriever handle diverse subject matter.

In [6]:
test_questions = [
    "How is Bayesian inference used in statistics?",
    "What work exists on portfolio optimization in quantitative finance?",
    "What are recent approaches to image classification using deep learning?",
    "What research has been done on black holes and gravitational waves?",
]

for q in test_questions:
    docs = retriever.invoke(q)
    show_results(q, docs)
    print("=" * 60)

QUESTION: How is Bayesian inference used in statistics?

--- Result 1 ---
Title: Bayesian Methods in Cosmology
Topics: Physics, Statistics
Snippet: Bayesian Methods in Cosmology

  These notes aim at presenting an overview of Bayesian statistics, the
underlying concepts and application methodology that will be useful to
astronomers seeking to ana ...

--- Result 2 ---
Title: The Frechet distribution: Estimation and Application an Overview
Topics: Statistics
Snippet: The Frechet distribution: Estimation and Application an Overview

  In this article, we consider the problem of estimating the parameters of the
Fréchet distribution from both frequentist and Bayesian ...

--- Result 3 ---
Title: How proper are Bayesian models in the astronomical literature?
Topics: Physics
Snippet: How proper are Bayesian models in the astronomical literature?

  The well-known Bayes theorem assumes that a posterior distribution is a
probability distribution. However, the posterior distribution  ...

QUEST

## Cell 7 — Test an edge case: a question unrelated to anything in the dataset

Our corpus is entirely research paper abstracts (CS, Physics, Math, Statistics, Quant Bio, Quant Finance). What happens when we ask something completely outside that scope? The retriever will still return its top-k *closest* documents — even if none of them are genuinely relevant — because similarity search always returns *something*, not "no result." This is an important limitation to understand and be ready to explain: **the retriever alone cannot tell you when nothing relevant exists** — that judgment has to come from the LLM (Stage 8) or from a distance-score threshold, not from the retriever itself.

In [7]:
off_topic_question = "What is the best recipe for making butter chicken?"

docs = retriever.invoke(off_topic_question)
show_results(off_topic_question, docs)

# Also show the distances, to see just how "far" these matches really are
results_with_scores = vectorstore.similarity_search_with_score(off_topic_question, k=3)
print("Distances for the off-topic question (expect these to be noticeably higher than Cell 5's):")
for doc, score in results_with_scores:
    print(f"Distance: {score:.4f}  |  Title: {doc.metadata['title']}")

QUESTION: What is the best recipe for making butter chicken?

--- Result 1 ---
Title: Towards "AlphaChem": Chemical Synthesis Planning with Tree Search and Deep Neural Network Policies
Topics: Computer Science, Physics
Snippet: Towards "AlphaChem": Chemical Synthesis Planning with Tree Search and Deep Neural Network Policies

  Retrosynthesis is a technique to plan the chemical synthesis of organic
molecules, for example dru ...

--- Result 2 ---
Title: Phase-tunable Josephson thermal router
Topics: Physics
Snippet: Phase-tunable Josephson thermal router

  Since the the first studies of thermodynamics, heat transport has been a
crucial element for the understanding of any thermal system. Quantum mechanics
has in ...

--- Result 3 ---
Title: Nutritionally recommended food for semi- to strict vegetarian diets based on large-scale nutrient composition data
Topics: Computer Science, Quantitative Biology
Snippet: Nutritionally recommended food for semi- to strict vegetarian diets based on 

## What to check after running this notebook

- **Cell 2:** confirms 20971 vectors reloaded successfully — no need to rebuild anything from Stage 6.
- **Cell 4:** do the 3 retrieved papers actually look relevant to "transformer efficiency"?
- **Cell 5:** note the distance values — these are your baseline for what a "good match" distance looks like.
- **Cell 6:** spot-check a couple of the 4 test questions — do the retrieved titles make sense for each topic?
- **Cell 7:** compare the off-topic question's distances to Cell 5's. They should be noticeably *higher* (less similar) — even though the retriever still confidently returns 3 documents regardless. This is worth remembering for Stage 8, where our prompt design needs to explicitly instruct the LLM to say when context is insufficient, rather than trusting the retriever to signal that on its own.

Paste back: the reload confirmation (Cell 2), the 3 titles + distances for the main test question (Cells 4-5), a couple of results from Cell 6, and the off-topic distances from Cell 7 — then we'll move to Stage 8 (the actual RAG chain).